# Lecture 3: Data Retrieval

:::{admonition} Learning Objectives
:class: tip
After this lecture, you will be able to:
- Explain why efficient data retrieval matters at scale
- Choose appropriate data formats (CSV vs Parquet) and data types
- Write performant queries using Polars and DuckDB
- Understand when to use in-memory processing vs databases
- Interact with relational databases from Python
- Navigate the data storage and management landscape
:::

## 1. Motivation: Working with (Biggish) Data

Data scientists spend most of their time on data — collecting, cleaning, and organising it.
If you do not use the right tools, you will spend most of your time **waiting** on data.

Two motivating case studies:
- Automating a business process for an insurance company (>20 tables, >500M rows, >300 columns)
- Training an LLM (Llama 3.1: 15 trillion tokens of pre-training data)

$ightarrow$ Being able to handle large amounts of data efficiently is a key skill.

## 2. How to Speed Things Up

We will load >178 million rows in <2 seconds. The key principles:
1. Only use what you need
2. Choose the right data types
3. Optimise your queries
4. Understand in-memory vs on-disk trade-offs
5. Use appropriate database management systems

### 2.1 Why You Should Not Work with CSV

:::{warning}
CSV files are human-readable but problematic at scale:
- No type information (everything is text)
- No compression
- Must read entire file to query a subset
- Parsing is slow and error-prone
:::

In [1]:
# TODO: Demonstrate CSV vs Parquet loading times
# import polars as pl
# import time
#
# # CSV loading
# tic = time.time()
# df_csv = pl.read_csv("large_dataset.csv")
# print(f"CSV load time: {time.time() - tic:.2f}s")
#
# # Parquet loading
# tic = time.time()
# df_parquet = pl.read_parquet("large_dataset.parquet")
# print(f"Parquet load time: {time.time() - tic:.2f}s")

### 2.2 Only Use What You Need

Column projection and row filtering at read time dramatically reduce I/O.

In [2]:
# TODO: Demonstrate column selection and row filtering at read time
# import polars as pl
#
# # Only read specific columns
# df = pl.scan_parquet("data.parquet").select(["col_a", "col_b"]).collect()
#
# # Filter rows during scan (predicate pushdown)
# df = pl.scan_parquet("data.parquet").filter(pl.col("year") > 2020).collect()

### 2.3 Choose the Right Data Types

Using appropriate data types reduces memory usage and speeds up computation:
-  vs  vs 
-  vs 
- Categorical vs string for repeated values
- Date/datetime types vs string parsing

In [3]:
# TODO: Demonstrate memory savings from appropriate dtypes
# import polars as pl
# import numpy as np
#
# # Compare memory usage
# df = pl.DataFrame({"id": range(1_000_000)})
# print(f"int64: {df.estimated_size() / 1e6:.1f} MB")
# df_small = df.cast({"id": pl.Int32})
# print(f"int32: {df_small.estimated_size() / 1e6:.1f} MB")

### 2.4 Optimise Your Queries

Lazy evaluation enables query optimisation:
- Predicate pushdown (filter early)
- Projection pushdown (select columns early)
- Common subexpression elimination

In [4]:
# TODO: Demonstrate lazy vs eager evaluation in Polars
# import polars as pl
#
# # Lazy evaluation with query plan optimisation
# query = (
#     pl.scan_parquet("nyc_taxi/*.parquet", hive_partitioning=True)
#     .group_by("passenger_count")
#     .agg(pl.mean("tip_amount").alias("mean_tip"))
#     .sort("passenger_count")
# )
#
# # Inspect the optimised query plan
# print(query.explain())
#
# # Execute
# result = query.collect()

### 2.5 In-Memory vs On-Disk

| Approach | Pros | Cons |
|----------|------|------|
| In-memory (Polars/pandas) | Fast for small-medium data | Limited by RAM |
| On-disk (DuckDB/SQLite) | Handles data larger than RAM | Slightly more overhead |

### 2.6 Relational Database Management Systems & SQL

Why databases?
- Data integrity (constraints, transactions)
- Concurrent access
- Query optimisation built-in
- Standardised query language (SQL)

In [5]:
# TODO: DuckDB example — loading and querying large data
# import duckdb
#
# con = duckdb.connect("nyc_taxi.ddb")
# query = """
#     SELECT
#         passenger_count,
#         AVG(tip_amount) AS mean_tip
#     FROM main.nyc_taxi AS t0
#     GROUP BY 1
# """
#
# import time
# tic = time.time()
# result = con.sql(query).show()
# print(f"Time: {time.time() - tic:.3f}s")

## 3. Python and Databases

Key libraries for database interaction:
- **DuckDB**: Embedded analytical database (OLAP), excellent for data science
- **SQLAlchemy**: ORM and database toolkit for production systems
- **sqlite3**: Built-in Python module for lightweight storage

In [6]:
# TODO: SQLAlchemy connection example
# from sqlalchemy import create_engine, text
#
# engine = create_engine("sqlite:///example.db")
# with engine.connect() as conn:
#     result = conn.execute(text("SELECT * FROM users LIMIT 5"))
#     for row in result:
#         print(row)

## 4. Data Storage & Management Landscape

Overview of storage options:

| Format/System | Type | Best For |
|--------------|------|----------|
| CSV | File | Small data, interop |
| Parquet | File | Columnar analytics |
| SQLite | Embedded DB | Local apps, prototyping |
| DuckDB | Embedded DB | Analytics, data science |
| PostgreSQL | Server DB | Production systems |
| Delta Lake | Table format | Versioned data lakes |

## 5. Repository Structure: .gitignore, Pre-commit, Type Hints

As our projects grow, we need better tooling:
- : Keep large data files out of version control
- Pre-commit hooks: Automate code quality checks
- Type hints: Document expected types for better IDE support and error catching

In [7]:
# TODO: Type hints example
# def load_data(path: str, *, columns: list[str] | None = None) -> pl.DataFrame:
#     """Load a parquet file with optional column selection."""
#     if columns is not None:
#         return pl.read_parquet(path, columns=columns)
#     return pl.read_parquet(path)

:::{admonition} Key Takeaways
:class: important
- Choose Parquet over CSV for any non-trivial dataset
- Use lazy evaluation and query optimisation (Polars, DuckDB)
- Pick the right tool: in-memory for speed, databases for scale and integrity
- Right-size your data types to reduce memory footprint
- Keep large data out of Git; use  and data registries
:::